# v2 — 05: Iteration Comparison (v1 vs v2)

This notebook documents the **iterative development** of the dynasty fantasy football model.
It compares v1 and v2 side-by-side to show what we gained by adding rookies, injury history,
draft capital, combine athleticism, and college production.

## What Changed Between v1 and v2

| Dimension | v1 | v2 |
|---|---|---|
| **Rookies** | Excluded (no prior PPR → dropped) | Included (`is_rookie=1`, NFL lags=0) |
| **Injury data** | None | `games_missed_prev1`, `ir_flag_prev1`, `career_games_missed_rate` |
| **Draft capital** | None | `draft_pick`, `draft_round`, `age_at_draft` |
| **Combine** | None | `combine_forty`, `combine_weight`, `combine_vertical` |
| **College production** | None | `college_rec_yards`, `college_rec_tds`, `college_dominator_rate` |
| **Feature count** | ~19 | ~40 |
| **Dataset size** | Veterans only | Veterans + Rookies |

**Prerequisite:** Run `v1_03_pipeline_cleaning.ipynb` and `v2_03_pipeline_cleaning.ipynb`
to populate both `model_ready` tables (you may need to rename one to `model_ready_v1`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, text
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.models.predictor import (
    load_model_data, train_and_evaluate, get_feature_importance, FEATURE_COLS,
    TRAIN_END, VAL_YEAR, TEST_YEAR,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)
%matplotlib inline

## 1. Hardcoded v1 Results

v1 results captured from `v1_04_modeling.ipynb` output.
These are the baseline numbers before any v2 enhancements.

In [ ]:
# v1 validation results (from v1_04_modeling.ipynb output)
v1_results = [
    {'model': 'Naive Baseline (prev year)', 'mae': 69.05, 'rmse': 94.00, 'r2': -0.077},
    {'model': 'Linear Regression',          'mae': 46.31, 'rmse': 60.66, 'r2':  0.551},
    {'model': 'Ridge Regression',           'mae': 46.25, 'rmse': 60.59, 'r2':  0.552},
    {'model': 'Random Forest',              'mae': 46.63, 'rmse': 61.32, 'r2':  0.542},
    {'model': 'Gradient Boosting',          'mae': 47.43, 'rmse': 62.23, 'r2':  0.528},
    {'model': 'XGBoost',                    'mae': 46.77, 'rmse': 61.14, 'r2':  0.544},
]

v1_best_model  = 'Ridge Regression'
v1_best_val_mae = 46.25
v1_test_mae    = 48.26
v1_test_rmse   = 63.87
v1_test_r2     = 0.513
v1_dataset_size = 4058  # total rows in model_ready
v1_features    = 19      # features used
v1_rookies     = 0       # rookies in dataset

v1_df = pd.DataFrame(v1_results).sort_values('mae')
print('v1 Validation Results:')
print(v1_df.to_string(index=False))
print(f'\nv1 Best on validation: {v1_best_model} (MAE={v1_best_val_mae})')
print(f'v1 Test MAE: {v1_test_mae} | RMSE: {v1_test_rmse} | R²: {v1_test_r2}')

## 2. v1 Gaps Identified

After reviewing v1 results, three major gaps were identified for dynasty use:

In [ ]:
gaps = {
    'Rookies excluded': (
        'Dynasty values rookie upside highly. v1 dropped every player with no prior NFL season '
        'because ppr_pts_prev1 was null. This means no rookie could ever appear in dynasty rankings.'
    ),
    'No injury signal': (
        'v1 used games_played as a proxy but had no explicit injury history. '
        'Knowing a player was on IR last year (or missed 8+ games) is a strong durability signal.'
    ),
    'No pre-NFL data': (
        'Draft capital, combine athleticism, and college production are all known before a player '
        'takes their first NFL snap. For rookies these are the primary predictors, but v1 had none.'
    ),
}

for gap, reason in gaps.items():
    print(f'[GAP] {gap}')
    print(f'      {reason}')
    print()

## 3. Run v2 Model

Load the v2 `model_ready` table (built by `v2_03_pipeline_cleaning.ipynb`) and train all models.

In [ ]:
df = load_model_data()
print(f'v2 dataset shape: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')

if 'is_rookie' in df.columns:
    v2_rookies = int(df['is_rookie'].sum())
    print(f'Rookies: {v2_rookies} | Veterans: {len(df) - v2_rookies}')
else:
    v2_rookies = 0

available_v2 = [c for c in FEATURE_COLS if c in df.columns]
print(f'Features available: {len(available_v2)}')

In [ ]:
v2_results_raw, pred_df = train_and_evaluate(df)

In [ ]:
v2_df = pd.DataFrame(v2_results_raw['results']).sort_values('mae')
v2_best_model = v2_results_raw['best_model']
v2_best_val_mae = v2_df[v2_df['model'] == v2_best_model]['mae'].values[0]
print('v2 Validation Results:')
print(v2_df.to_string(index=False))
print(f'\nv2 Best on validation: {v2_best_model} (MAE={v2_best_val_mae:.2f})')

## 4. Side-by-Side Comparison

In [ ]:
# Merge v1 and v2 results on model name for head-to-head
comparison = v1_df.merge(v2_df, on='model', suffixes=('_v1', '_v2'))
comparison['mae_delta'] = comparison['mae_v2'] - comparison['mae_v1']
comparison['r2_delta']  = comparison['r2_v2']  - comparison['r2_v1']
comparison_sorted = comparison.sort_values('mae_v1')

display_cols = ['model', 'mae_v1', 'mae_v2', 'mae_delta', 'r2_v1', 'r2_v2', 'r2_delta']
print('Side-by-side validation MAE and R² (delta = v2 - v1, negative MAE delta = improvement):')
print(comparison_sorted[[c for c in display_cols if c in comparison_sorted.columns]].to_string(index=False))

In [ ]:
models_shared = comparison['model'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE comparison
ax = axes[0]
x = np.arange(len(models_shared))
w = 0.35
ax.barh(x - w/2, comparison['mae_v1'], w, label='v1', color='steelblue', alpha=0.85)
ax.barh(x + w/2, comparison['mae_v2'], w, label='v2', color='darkorange', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(comparison['model'])
ax.invert_xaxis()
ax.set_xlabel('MAE (lower = better)')
ax.set_title('Validation MAE: v1 vs v2')
ax.legend()

# R² comparison
ax = axes[1]
ax.barh(x - w/2, comparison['r2_v1'], w, label='v1', color='steelblue', alpha=0.85)
ax.barh(x + w/2, comparison['r2_v2'], w, label='v2', color='darkorange', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(comparison['model'])
ax.set_xlabel('R² (higher = better)')
ax.set_title('Validation R²: v1 vs v2')
ax.legend()

plt.suptitle('v1 vs v2 Model Performance — Validation Set', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 5. Dataset Size Comparison

In [ ]:
v2_dataset_size = len(df)
v2_features_count = len(available_v2)

size_comparison = pd.DataFrame({
    'Metric': [
        'Dataset rows', 'Rookies included', 'Features used',
        'Best val MAE', 'Best val model',
    ],
    'v1': [v1_dataset_size, v1_rookies, v1_features, v1_best_val_mae, v1_best_model],
    'v2': [v2_dataset_size, v2_rookies, v2_features_count, f'{v2_best_val_mae:.2f}', v2_best_model],
})

print('Dataset and model summary:')
print(size_comparison.to_string(index=False))

## 6. Feature Importance Comparison

In [ ]:
# v1 top features (hardcoded from v1_04_modeling.ipynb output)
v1_top_features = [
    ('fantasy_points_ppr', 0.195),
    ('ppr_pts_prev1',      0.178),
    ('ppr_per_game',       0.142),
    ('age_at_season',      0.098),
    ('target_share',       0.071),
    ('receiving_epa',      0.054),
    ('ppr_pts_prev2',      0.048),
    ('avg_snap_pct',       0.041),
    ('td_rate',            0.037),
    ('air_yards_share',    0.031),
]
v1_feat_df = pd.DataFrame(v1_top_features, columns=['feature', 'importance'])

# v2 feature importance — computed live
print('Computing v2 feature importances (Random Forest on train set)...')
v2_feat_df = get_feature_importance(df)
print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# v1 feature importance
ax = axes[0]
ax.barh(v1_feat_df['feature'][::-1], v1_feat_df['importance'][::-1], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('v1 — Top 10 Feature Importances\n(veterans only, no draft/college/injury)')

# v2 feature importance
ax = axes[1]
top20 = v2_feat_df.head(20)
ax.barh(top20['feature'][::-1], top20['importance'][::-1], color='darkorange')
ax.set_xlabel('Importance')
ax.set_title('v2 — Top 20 Feature Importances\n(veterans + rookies, all new features)')

plt.suptitle('Feature Importance: v1 vs v2 Random Forest', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

print('\nv2 top 20 features:')
print(v2_feat_df.head(20).to_string(index=False))

## 7. New Features — Did They Help?

Check which v2-only features rank highly — this validates the data engineering effort.

In [ ]:
v2_only_features = [
    'is_rookie',
    'games_missed_prev1', 'career_games_missed_rate', 'ir_flag_prev1',
    'draft_pick', 'draft_round', 'age_at_draft', 'is_undrafted',
    'combine_forty', 'combine_weight', 'combine_height', 'combine_vertical', 'combine_bench',
    'college_rec_yards', 'college_rec_tds', 'college_targets',
    'college_rush_yards', 'college_rush_tds', 'college_dominator_rate',
]

new_feat_importances = v2_feat_df[v2_feat_df['feature'].isin(v2_only_features)].copy()
new_feat_importances['rank'] = v2_feat_df.reset_index(drop=True).index[
    v2_feat_df['feature'].isin(v2_only_features)
].tolist()

print('v2-only features and their importance rank:')
for _, row in new_feat_importances.sort_values('importance', ascending=False).iterrows():
    bar = '|' * int(row['importance'] * 500)
    print(f'  {row["feature"]:35s} importance={row["importance"]:.4f}  {bar}')

## 8. Rookie-Specific Performance

v1 had no rookies. v2 now includes them — how well does the model predict rookie output?

In [ ]:
if 'is_rookie' in pred_df.columns:
    from sklearn.metrics import mean_absolute_error

    rookies_pred = pred_df[pred_df['is_rookie'] == 1].dropna(
        subset=['actual_ppr_next', 'predicted_ppr_next']
    )
    vets_pred = pred_df[pred_df['is_rookie'] == 0].dropna(
        subset=['actual_ppr_next', 'predicted_ppr_next']
    )

    if len(rookies_pred) > 0:
        rmae = mean_absolute_error(rookies_pred['actual_ppr_next'], rookies_pred['predicted_ppr_next'])
        vmae = mean_absolute_error(vets_pred['actual_ppr_next'], vets_pred['predicted_ppr_next'])
        print(f'Test-set MAE breakdown:')
        print(f'  Rookie MAE:   {rmae:.2f} (n={len(rookies_pred)})')
        print(f'  Veteran MAE:  {vmae:.2f} (n={len(vets_pred)})')
        print(f'  Overall MAE:  {mean_absolute_error(pred_df["actual_ppr_next"], pred_df["predicted_ppr_next"]):.2f}')
    else:
        print('No rookies in test set (2023 rookies need a 2024 season to have a target — check data range)')
else:
    print('is_rookie column not found in pred_df')

In [ ]:
if 'is_rookie' in pred_df.columns and len(pred_df[pred_df['is_rookie']==1]) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, (label, sub, color) in zip(axes, [
        ('Rookies', rookies_pred, 'darkorange'),
        ('Veterans', vets_pred, 'steelblue'),
    ]):
        ax.scatter(sub['actual_ppr_next'], sub['predicted_ppr_next'],
                   alpha=0.4, s=25, color=color)
        lim = max(sub['actual_ppr_next'].max(), sub['predicted_ppr_next'].max()) * 1.05
        ax.plot([0, lim], [0, lim], 'r--', alpha=0.5, label='Perfect')
        mae_val = mean_absolute_error(sub['actual_ppr_next'], sub['predicted_ppr_next'])
        ax.set_title(f'{label} — Actual vs Predicted\nMAE={mae_val:.1f}')
        ax.set_xlabel('Actual PPR Points (next season)')
        ax.set_ylabel('Predicted PPR Points')
        ax.legend()

    plt.suptitle('v2 Prediction Quality: Rookies vs Veterans (Test Set)', y=1.02)
    plt.tight_layout()
    plt.show()

## 9. Top Rookie Dynasty Rankings (v2 Test Set)

In [ ]:
if 'is_rookie' in pred_df.columns:
    display_cols = ['predicted_rank', 'full_name', 'position', 'is_rookie',
                    'predicted_ppr_next', 'actual_ppr_next', 'error']
    available_display = [c for c in display_cols if c in pred_df.columns]

    rookie_rankings = pred_df[pred_df['is_rookie'] == 1].head(20)
    if len(rookie_rankings) > 0:
        print('=== Top 20 Rookie Dynasty Rankings (v2, Test Set) ===')
        print(rookie_rankings[available_display].to_string(index=False))
    else:
        print('No rookies in test set — rookies from 2023 class need 2024 actuals as their target.')
        print('The rookies ARE in the training set (2016-2021 classes) and validation (2022 class).')
        
        # Show rookies from validation year instead
        val_rookies = df[(df['season'] == VAL_YEAR) & (df.get('is_rookie', 0) == 1)]
        if len(val_rookies) > 0:
            print(f'\nRookies in validation year ({VAL_YEAR}):')
            show_cols = ['full_name', 'position', 'draft_pick', 'college_rec_yards', 'ppr_pts_next']
            print(val_rookies[[c for c in show_cols if c in val_rookies.columns]].head(15).to_string(index=False))

## 10. Summary: What v2 Achieved

In [ ]:
print('=' * 60)
print('ITERATION SUMMARY: v1 → v2')
print('=' * 60)

print(f'\n[DATA]')
print(f'  Dataset rows:    {v1_dataset_size:>6d} → {len(df):>6d} ({len(df)-v1_dataset_size:+d})')
print(f'  Rookies:         {v1_rookies:>6d} → {v2_rookies:>6d} (dynasty-relevant addition)')
print(f'  Features:        {v1_features:>6d} → {len(available_v2):>6d}')

print(f'\n[PERFORMANCE — Validation Set]')
print(f'  v1 best model:   {v1_best_model} (MAE={v1_best_val_mae:.2f})')
print(f'  v2 best model:   {v2_best_model} (MAE={v2_best_val_mae:.2f})')
delta = v2_best_val_mae - v1_best_val_mae
direction = 'IMPROVED' if delta < 0 else 'DEGRADED'
print(f'  MAE delta:       {delta:+.2f} ({direction})')

print(f'\n[KEY WINS]')
print(f'  - Rookies now scoreable — critical for dynasty draft rankings')
print(f'  - Draft capital and college production carry signal for unproven players')
print(f'  - Injury history adds durability dimension missing from v1')

print(f'\n[KNOWN LIMITATIONS / v3 IDEAS]')
print(f'  - College data coverage: only players with cfb_player_id in draft picks get college stats')
print(f'  - Undrafted free agents: no draft capital, sparse combine — model relies on NFL stats')
print(f'  - Model regresses to mean for breakout players (large errors on Ja'\''Marr Chase, Lamar Jackson 2023)')
print(f'  - Position-specific models could improve TE and RB predictions independently')
print(f'  - Multi-year targets (2yr, 3yr cumulative PPR) better capture dynasty value than 1yr ahead')
print(f'  - ADP/market value as a feature or calibration target')
print('=' * 60)

## 11. v3 Roadmap

Based on v2 results, these are the highest-leverage improvements for the next iteration:

| Priority | Feature / Change | Expected Impact |
|---|---|---|
| High | **Multi-year target** — predict cumulative 2-3 yr PPR | Better dynasty signal; less noise from single-year variance |
| High | **Position-specific models** — separate QB/RB/WR/TE | Each position has different age curves and usage patterns |
| Medium | **Dynasty ADP integration** — use market price as feature | Calibrates model output against consensus; identifies value |
| Medium | **Team context features** — OL quality, target depth chart | Players on high-volume offenses have a structural advantage |
| Low | **Improve college join rate** — fuzzy name matching fallback | Many players lack `cfb_player_id`; better join = more college data |
| Low | **UDFA-specific model** — separate treatment for undrafted | UDFAs lack draft capital; they earn a spot on pure NFL production |
